# 02. Shared dataset

We prepare one dataset for all models. Each row is one student in one year. The predictors are Moodle activity features from the first 8 weeks of the semester and the target is the final numerical grade.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("tcs_dataset")

# prediction point: 8 weeks after the start
CUTOFF_WEEKS = 8
TRAIN_YEARS = [2019, 2021, 2022, 2023, 2024]
TEST_YEARS = [2025, 2026]
print(DATA_DIR.exists())

True


## 1. Load student events

We load the Moodle events of all years and keep only the columns we need and only the students (actor_id starts with S).

In [2]:
frames = []
for year in range(2019, 2027):
    path = DATA_DIR / "data" / str(year) / "events.parquet"
    df = pd.read_parquet(path, columns=["actor_id", "event_time", "event_name"])
    df["year"] = year
    frames.append(df)

events = pd.concat(frames, ignore_index=True)
events = events[events["actor_id"].astype(str).str.startswith("S")]
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)
print(events.shape)

(733090, 4)


## 2. Start dates of the courses

`course_runs.csv` has the start date of each course year. We need it to find the prediction point.

In [3]:
course_runs = pd.read_csv(DATA_DIR / "data" / "course_runs.csv")
course_runs["year"] = range(2019, 2027)
course_runs["course_start_date"] = pd.to_datetime(course_runs["course_start_date"], utc=True)
start_dates = course_runs.set_index("year")["course_start_date"]
start_dates

year
2019   2019-02-11 00:00:00+00:00
2020   2020-02-10 00:00:00+00:00
2021   2021-02-08 00:00:00+00:00
2022   2022-02-07 00:00:00+00:00
2023   2023-02-06 00:00:00+00:00
2024   2024-02-12 00:00:00+00:00
2025   2025-02-10 00:00:00+00:00
2026   2026-02-09 00:00:00+00:00
Name: course_start_date, dtype: datetime64[us, UTC]

## Final grades

We load `grades.csv` of each year. The target is `course_numeric_grade`. We keep only students with a known grade and exclude 2020 (no grades).

In [4]:
test_grades = pd.read_csv(DATA_DIR / "data" / "2021" / "grades.csv")
print(test_grades.shape)
print(test_grades.columns.tolist())

(142, 41)
['course_run_id', 'actor_id', 'homework_01_eng', 'homework_01_est', 'homework_02_eng', 'homework_02_est', 'homework_03_eng', 'homework_03_est', 'homework_04_eng', 'homework_04_est', 'homework_05_eng', 'homework_05_est', 'homework_06_eng', 'homework_06_est', 'homework_07_eng', 'homework_07_est', 'homework_08_eng', 'homework_08_est', 'homework_09_eng', 'homework_09_est', 'homework_10_eng', 'homework_10_est', 'homework_11_eng', 'homework_11_est', 'homework_12_eng', 'homework_12_est', 'midterm_attempt_1_eng', 'midterm_attempt_1_est', 'midterm_attempt_2_eng', 'midterm_attempt_2_est', 'midterm_attempt_3_eng', 'midterm_attempt_3_est', 'final_exam_attempt_1_eng', 'final_exam_attempt_1_est', 'final_exam_attempt_2_eng', 'final_exam_attempt_2_est', 'final_exam_attempt_3_eng', 'final_exam_attempt_3_est', 'bonus', 'course_numeric_grade', 'course_letter_grade']


In [5]:
frames = []
for year in range(2019, 2027):
    g = pd.read_csv(DATA_DIR / "data" / str(year) / "grades.csv")
    g["year"] = year
    frames.append(g[["year", "actor_id", "course_numeric_grade"]])

grades = pd.concat(frames, ignore_index=True)
grades["course_numeric_grade"] = pd.to_numeric(grades["course_numeric_grade"], errors="coerce")

grades = grades[grades["course_numeric_grade"].notna()]
grades = grades[grades["year"] != 2020]

print("duplicate student-years:", grades.duplicated(["year", "actor_id"]).sum())
grades.groupby("year")["course_numeric_grade"].agg(["count", "mean", "std", "min", "max"]).round(1)

duplicate student-years: 0


,count,mean,std,min,max
year,,,,,
2019,112,65.9,21.6,2.0,109.2
2021,137,70.5,27.2,0.0,110.0
2022,79,73.0,20.7,3.7,110.0
2023,104,58.5,27.7,0.0,106.3
2024,119,67.1,24.2,0.0,110.0
2025,137,59.2,25.8,0.0,110.0
2026,142,70.4,27.2,0.0,111.2


## 4. Keep only events before the prediction point

For each year the prediction point is 8 weeks after the start of the course. Later events must not be used, otherwise the model would see information from the future.

In [6]:
events["cutoff"] = events["year"].map(start_dates) + pd.Timedelta(weeks=CUTOFF_WEEKS)
early = events[events["event_time"] < events["cutoff"]].copy()

print("events before the cutoff:", len(early), "of", len(events))
early.groupby("year").size()

events before the cutoff: 218217 of 733090


year
2019    19864
2020    23010
2021    40862
2022    19532
2023    30816
2024    41995
2025    22502
2026    19636
dtype: int64

In [7]:
# Count the most common Moodle activities in the first 8 weeks
event_counts = early["event_name"].value_counts()

display(event_counts.head(30))

event_name
Course viewed                                    56634
Course module viewed                             56102
Quiz attempt viewed                              34478
Quiz attempt updated                              8817
Quiz attempt reviewed                             7982
Grade user report viewed                          6168
The status of the submission has been viewed.     5945
BigBlueButton activity viewed                     4861
Quiz attempt summary viewed                       4796
User graded                                       4437
Quiz attempt started                              4384
Course user report viewed                         3766
Discussion updated                                3746
Quiz attempt submitted                            3548
BigBlueButton session joined                      2781
Recording viewed                                  2439
Submission created.                                970
Course module instance list viewed                 960

## 5. Activity features per student and year

- total_events: number of events
- active_days: number of different days with at least one event
- course_module_views: number of "Course module viewed" events
- course_views: number of "Course viewed" events

In [8]:
early["date"] = early["event_time"].dt.tz_convert("Europe/Tallinn").dt.date
early["is_module_view"] = (early["event_name"] == "Course module viewed").astype(int)
early["is_course_view"] = (early["event_name"] == "Course viewed").astype(int)

features = early.groupby(["year", "actor_id"]).agg(
    total_events=("event_name", "size"), 
    active_days=("date", "nunique"), 
    course_module_views=("is_module_view", "sum"), 
    course_views=("is_course_view", "sum"), 
).reset_index()

features.head()

,year,actor_id,total_events,active_days,course_module_views,course_views
0,2019,S027cf9,132,18,20,34
1,2019,S032255,48,6,18,8
2,2019,S03a8b8,221,30,39,75
3,2019,S04cf7f,161,20,26,38
4,2019,S05f285,456,32,76,177


## 6. Combine features and grades

We start from the students with a known grade. Students without any activity before the prediction point get zeros.

In [9]:
feature_cols = ["total_events", "active_days", "course_module_views", "course_views"]

data = grades.merge(features, on=["year", "actor_id"], how="left")
data[feature_cols] = data[feature_cols].fillna(0).astype(int)

print(data.shape)
print("students without activity before the cutoff:", (data["total_events"] == 0).sum())
data.groupby("year").size()

(830, 7)
students without activity before the cutoff: 1


year
2019    112
2021    137
2022     79
2023    104
2024    119
2025    137
2026    142
dtype: int64

## 7. Train/test split and saving

Train: 2019 and 2021-2024. Test: 2025 and 2026 (the same for all models).

In [10]:
train = data[data["year"].isin(TRAIN_YEARS)]
test = data[data["year"].isin(TEST_YEARS)]

print("train:", train.shape, " test:", test.shape)
print(test.groupby("year").size())

train.to_csv("shared_train.csv", index=False)
test.to_csv("shared_test.csv", index=False)

train: (551, 7)  test: (279, 7)
year
2025    137
2026    142
dtype: int64


## Notes

We exclude 2020 because there are no grades. Students without a final grade are not in the dataset. Only events before the prediction point (8 weeks after the start of the course) are used.